Title: ST-554 Homework 10 \
Author: Stephen Griggs: \
Date: 4/16/2026

# Introduction
This notebook explores Spark Structured Streaming in two parts:

1. **Streaming with the `rate` source:**
    - Generate a stream using the built-in `rate` format.
    - Apply transformations to the `value` column using `pyspark.sql.functions`:
        - Square root of `value`
        - `value` mod 4
    - Write the results to an in-memory table via `writeStream`.
    - Let the query run briefly, then inspect the accumulated output.

2. **Streaming CSV files through a fitted Pipeline:**
    - Read `bikeDetails_for_fit.csv` into a Spark DataFrame.
    - Build and fit a Pipeline with two stages:
        - `SQLTransformer` for log transforms and a one-owner dummy variable.
        - `VectorAssembler` to assemble the features column.
    - Set up a file-based `readStream` to watch a folder for incoming `bikeDetails_add*.csv` files.
    - Transform each new file using the fitted pipeline and append the results to the console.

## Part 1: Streaming with the `rate` Source

First, create a streaming DataFrame using the `rate` format, apply `sqrt` and `mod 4` transformations to the `value` column, then write the results to an in-memory table. After letting the query run for ~30 seconds, it's stopped and the accumulated output is displayed.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sqrt, col
import time

spark = SparkSession.builder.appName("HW10_Streaming").getOrCreate()

# Quiet down Spark's WARN messages and streaming checkpoint/AQE warnings for cleaner output
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
spark.conf.set("spark.sql.adaptive.enabled", "false")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 17:18:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Set up the rate stream
rate_stream = spark.readStream.format("rate").load()

# Apply transformations: sqrt(value) and value mod 4
transformed = rate_stream.select(
    col("timestamp"),
    col("value"),
    sqrt(col("value")).alias("sqrt_value"),
    (col("value") % 4).alias("mod4_value")
)

# Write to memory and start the query
query = (transformed.writeStream
         .format("memory")
         .queryName("rate_table")
         .start())

# Let it run ~30 seconds, then stop
time.sleep(30)
query.stop()

# Show the collected results
spark.sql("SELECT * FROM rate_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-17 17:18:...|    0|               0.0|         0|
|2026-04-17 17:18:...|    1|               1.0|         1|
|2026-04-17 17:18:...|    2|1.4142135623730951|         2|
|2026-04-17 17:18:...|    3|1.7320508075688772|         3|
|2026-04-17 17:18:...|    4|               2.0|         0|
|2026-04-17 17:18:...|    5|  2.23606797749979|         1|
|2026-04-17 17:18:...|    6| 2.449489742783178|         2|
|2026-04-17 17:18:...|    7|2.6457513110645907|         3|
|2026-04-17 17:18:...|    8|2.8284271247461903|         0|
|2026-04-17 17:18:...|    9|               3.0|         1|
|2026-04-17 17:18:...|   10|3.1622776601683795|         2|
|2026-04-17 17:18:...|   11|   3.3166247903554|         3|
|2026-04-17 17:18:...|   12|3.4641016151377544|         0|
|2026-04-17 17:18:...|   13| 3.605551275463989|         

## Part 2: Streaming CSV Files Through a Fitted Pipeline

Read `bikeDetails_for_fit.csv` into a Spark DataFrame, then build and fit a two-stage Pipeline:

1. `SQLTransformer`: log-transforms selling_price as a label and km_driven, then creates a one_owner dummy.
2. `VectorAssembler`: assembles year, log_km_driven, and one_owner into a features column.

Then set up a file-based readStream watching a folder for the `bikeDetails_add*.csv` files, apply the fitted pipeline's .transform() to each incoming file, and append the results to the console.

In [4]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler

# Read the fit CSV as a Spark DataFrame
bike_fit = spark.read.csv("st554-homework10/bikeDetails_for_fit.csv", header=True, inferSchema=True)

# Stage 1: SQL transform (log transforms, rename, dummy variable)
sql_trans = SQLTransformer(statement="""
    SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
           CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
    FROM __THIS__
""")

# Stage 2: Vector assembler for features
assembler = VectorAssembler(
    inputCols=["year", "log_km_driven", "one_owner"],
    outputCol="features"
)

# Build and fit the pipeline
pipeline = Pipeline(stages=[sql_trans, assembler])
fitted_pipeline = pipeline.fit(bike_fit)